# 7. SSCR supplementary robustness and final figures

Reproduces the post-preregistration **individual-level accuracy (Table S7)**, **pooled Iraq-exclusion sensitivity (Table S8)**, **flat normalized-dispersion profile benchmark**, and final publication-oriented versions of **Figures 2 and 4**.

Run from the repository root **after notebooks 1–5**. It uses the original WVS analysis sample and saved model predictions; there are **no API calls**, no fitting of new LLMs, and no redistribution of WVS microdata. It does not change the preregistered primary tests.

The country-adjusted normalized-SD regression used in the revised main Table 1 is already implemented in Notebook 3, section 5B, which writes 08a_country_FE_normalized_sd_slopes.csv. The RIF analysis remains in that notebook for the supplementary diagnostic (Table S6).

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import pearsonr, spearmanr

ROOT = Path("output/full_wvs")
PROFILE_PATH = ROOT / "profiles_for_prediction.csv"
PRED_DIR = ROOT / "predictions"
LOCKED = ROOT / "analysis_locked"
BENCHMARK = ROOT / "non_llm_benchmarks_exact_inputs"
OUT = ROOT / "sscr_additional_robustness"
OUT.mkdir(parents=True, exist_ok=True)

PRED_FILES = {
    "GPT-5.6 Luna": "openai_gpt56_luna_FULL.csv",
    "Claude Sonnet 5": "anthropic_sonnet5_FULL.csv",
    "Gemini 3.8 Flash": "google_gemini38_flash_FULL.csv",
    "DeepSeek V4.1 Flash": "deepseek_v41_flash_FULL.csv",
    "Gemma 4 31B": "deepinfra_gemma4_31b_FULL.csv",
    "Qwen 3.7 Plus": "alibaba_qwen37_plus_FULL.csv",
}
SOURCE_ORDER = ["Human", *PRED_FILES.keys()]
INCOME_GROUPS = np.arange(1, 11)
EXPECTED_N = 93901

for path in [PROFILE_PATH, *[PRED_DIR / name for name in PRED_FILES.values()]]:
    if not path.exists():
        raise FileNotFoundError(
            "Missing required input: " + str(path) +
            ". Run Notebooks 1 and 2 using locally obtained WVS data and saved model predictions."
        )
print("Output directory:", OUT)


In [ ]:
# Load the same 93,901 respondent records and join by their frozen IDs.
profiles = pd.read_csv(PROFILE_PATH, low_memory=False)
required = {"WVS_ROW_ID", "Q49", "Q288", "B_COUNTRY_ALPHA", "COUNTRY_NAME"}
assert required.issubset(profiles.columns), sorted(required - set(profiles.columns))
assert len(profiles) == EXPECTED_N
assert profiles["WVS_ROW_ID"].is_unique

data = profiles[["WVS_ROW_ID", "Q49", "Q288", "B_COUNTRY_ALPHA", "COUNTRY_NAME"]].copy()
data["Q49"] = pd.to_numeric(data["Q49"], errors="raise")
data["Q288"] = pd.to_numeric(data["Q288"], errors="raise").astype(int)
assert data["Q49"].between(1, 10).all()
assert data["Q288"].between(1, 10).all()
assert data["B_COUNTRY_ALPHA"].nunique() == 66

SOURCE_COLS = {"Human": "Q49"}
for idx, (model, filename) in enumerate(PRED_FILES.items(), start=1):
    p = pd.read_csv(PRED_DIR / filename, low_memory=False)
    assert {"WVS_ROW_ID", "prediction"}.issubset(p.columns)
    assert len(p) == EXPECTED_N and p["WVS_ROW_ID"].is_unique, model
    if "status" in p.columns:
        assert p["status"].eq("ok").all(), model
    col = "llm_" + str(idx)
    p[col] = pd.to_numeric(p["prediction"], errors="raise")
    assert p[col].between(1, 10).all(), model
    assert (p[col] == np.floor(p[col])).all(), model
    data = data.merge(p[["WVS_ROW_ID", col]], on="WVS_ROW_ID", how="left", validate="one_to_one")
    assert data[col].notna().all(), model
    SOURCE_COLS[model] = col

assert len(data) == EXPECTED_N and data["WVS_ROW_ID"].is_unique
print("Verified", len(data), "respondents,", data["B_COUNTRY_ALPHA"].nunique(),
      "countries, and", len(PRED_FILES), "LLM prediction files.")


## Table S7: Individual-level predictive accuracy

Pearson correlations, mean absolute error (MAE), and root-mean-square error (RMSE) are measured against the actual respondent-level life-satisfaction response. The sample-mean prediction is a descriptive baseline. OLS/Lasso accuracy is appended **only if** the out-of-sample prediction file from Notebook 5 is present; the main published Table S7 reports the six LLMs.

In [ ]:
truth = data["Q49"].to_numpy(float)
def metric_row(label, prediction):
    pred = np.asarray(prediction, float)
    assert len(pred) == len(truth) and np.isfinite(pred).all()
    error = pred - truth
    return {
        "Source": label,
        "Pearson_r": float(np.corrcoef(truth, pred)[0, 1])
            if np.std(pred) > 0 else np.nan,
        "MAE": float(np.mean(np.abs(error))),
        "RMSE": float(np.sqrt(np.mean(error ** 2))),
        "Overall_prediction_SD": float(np.std(pred, ddof=1)),
        "Overall_SD_ratio_to_human": float(np.std(pred, ddof=1) / np.std(truth, ddof=1))
    }

llm_accuracy = pd.DataFrame([
    metric_row(model, data[col].to_numpy(float))
    for model, col in SOURCE_COLS.items() if model != "Human"
])
llm_accuracy.to_csv(OUT / "Table_S7_LLM_individual_accuracy.csv", index=False)
display(llm_accuracy.round(4))

baseline = np.full(len(data), float(np.mean(truth)))
baseline_row = metric_row("Sample-mean baseline", baseline)
print("Sample-mean RMSE:", round(baseline_row["RMSE"], 4),
      "; human SD (population definition):", round(float(np.std(truth)), 4))

extended = [*llm_accuracy.to_dict(orient="records"), baseline_row]
oof_path = BENCHMARK / "02_OOF_predictions_OLS_Lasso.csv"
if oof_path.exists():
    oof = pd.read_csv(oof_path, low_memory=False)
    assert {"WVS_ROW_ID", "OLS", "Lasso"}.issubset(oof.columns)
    aligned = data[["WVS_ROW_ID"]].merge(
        oof[["WVS_ROW_ID", "OLS", "Lasso"]],
        on="WVS_ROW_ID", how="left", validate="one_to_one")
    assert aligned[["OLS", "Lasso"]].notna().all().all()
    extended.extend([
        metric_row("OLS (out of sample)", aligned["OLS"].to_numpy(float)),
        metric_row("Lasso (out of sample)", aligned["Lasso"].to_numpy(float))
    ])
else:
    print("Optional OLS/Lasso predictions not found; run Notebook 5 to append them.")
pd.DataFrame(extended).to_csv(OUT / "accuracy_with_optional_benchmarks.csv", index=False)


## Table S8: Pooled income-gradient sensitivity excluding Iraq

For both samples, calculate each source's within-income-group SDs, normalize by its own weighted mean using **human income-group shares in the relevant sample**, and fit an equally weighted linear trend over the ten groups. This matches the main pooled-profile estimand, not the separate country-fixed-effect regression or the cross-country gradient correlation.

In [ ]:
def pooled_normalized_profile(frame, label):
    group_shares = (
        frame["Q288"].value_counts(normalize=True)
        .reindex(INCOME_GROUPS).to_numpy(float)
    )
    if np.isnan(group_shares).any():
        raise ValueError("One or more income groups is empty for " + label)
    rows, profiles_by_source = [], {}
    for name, column in SOURCE_COLS.items():
        sd = (frame.groupby("Q288")[column]
              .std(ddof=1).reindex(INCOME_GROUPS).to_numpy(float))
        scale = float(np.dot(group_shares, sd))
        normalized = sd / scale
        slope = float(np.polyfit(INCOME_GROUPS, normalized, 1)[0])
        rows.append({"Source": name, "Sample": label, "Normalized_slope": slope,
                     "Weighted_within_group_SD": scale})
        profiles_by_source[name] = normalized
    return pd.DataFrame(rows), profiles_by_source

all_rows, all_profiles = pooled_normalized_profile(data, "All 66 countries")
# Use the country code/name to avoid assuming any particular two- or three-letter code.
without_iraq = data.loc[
    ~data["COUNTRY_NAME"].astype(str).str.strip().str.casefold().eq("iraq")
].copy()
assert len(without_iraq) < len(data), "Country name 'Iraq' not found; inspect WVS labels."
assert without_iraq["B_COUNTRY_ALPHA"].nunique() == 65
no_rows, no_profiles = pooled_normalized_profile(without_iraq, "Excluding Iraq (65)")
table_s8 = all_rows[["Source", "Normalized_slope"]].rename(
    columns={"Normalized_slope": "All_66"}
).merge(no_rows[["Source", "Normalized_slope"]].rename(
    columns={"Normalized_slope": "Excluding_Iraq_65"}), on="Source", validate="one_to_one")
table_s8["Difference_excluding_minus_all"] = (
    table_s8["Excluding_Iraq_65"] - table_s8["All_66"])
table_s8.to_csv(OUT / "Table_S8_pooled_Iraq_sensitivity.csv", index=False)
display(table_s8.round(4))

iraq = data.loc[data["COUNTRY_NAME"].astype(str).str.strip().str.casefold().eq("iraq")]
iraq_equals = float(np.mean(iraq["Q49"] == iraq["Q288"]))
print("Iraq respondents:", len(iraq), "; fraction Q49 equal to Q288:", round(iraq_equals, 4))


## Flat-profile benchmark

A flat normalized SD profile is a vector of ten ones. Its RMSD from the observed human normalized profile is a descriptive 'no across-income structure' benchmark, **not** a statistical test or an alternative generative model. Use the unweighted RMSD across ten income groups, matching Table S4.

In [ ]:
human_profile = all_profiles["Human"]
flat_rmsd = float(np.sqrt(np.mean((np.ones(10) - human_profile) ** 2)))
flat_rows = [{"Source": "Flat profile (all normalized SDs = 1)",
              "RMSD_from_human": flat_rmsd}]
for name in SOURCE_ORDER[1:]:
    flat_rows.append({"Source": name,
                      "RMSD_from_human": float(np.sqrt(
                          np.mean((all_profiles[name] - human_profile) ** 2)))})
flat_comparison = pd.DataFrame(flat_rows)
flat_comparison.to_csv(OUT / "flat_profile_and_LLM_RMSD.csv", index=False)
display(flat_comparison.round(4))


## Figure 2: Normalized income-dispersion profiles and human bootstrap band

Uses the exact 1,000-country-cluster bootstrap group-level confidence intervals saved by Notebook 3. They are **percentile bootstrap intervals**, not pointwise parametric SE bands; the figure is descriptive and the global profile Wald test remains the preregistered inferential test.

In [ ]:
norm_path = LOCKED / "03_primary_normalized_sd_profiles.csv"
band_path = LOCKED / "05_primary_normalized_profiles_bootstrap_ci.csv"
if norm_path.exists() and band_path.exists():
    norm = pd.read_csv(norm_path)
    bands = pd.read_csv(band_path)
    h = bands.loc[bands["source"].eq("Human")].sort_values("Q288")
    assert len(h) == 10
    fig, ax = plt.subplots(figsize=(9.2, 5.7))
    for name in SOURCE_ORDER:
        sub = norm.loc[norm["source"].eq(name)].sort_values("Q288")
        assert len(sub) == 10
        ax.plot(sub["Q288"], sub["normalized_sd"],
                marker="o" if name == "Human" else None,
                linewidth=2.5 if name == "Human" else 1.5,
                label=name)
    ax.fill_between(h["Q288"].to_numpy(float),
                    h["ci_low"].to_numpy(float), h["ci_high"].to_numpy(float),
                    alpha=0.18, label="Human 95% bootstrap CI")
    ax.axhline(1, color="0.55", linestyle=":", linewidth=1)
    ax.set_xticks(INCOME_GROUPS)
    ax.set_xlabel("Household income group (1 = lowest, 10 = highest)")
    ax.set_ylabel("Normalized SD of life satisfaction")
    ax.legend(frameon=False, fontsize=8, ncol=2, loc="best")
    fig.tight_layout()
    fig.savefig(OUT / "Figure_2_human_bootstrap_band.png",
                dpi=300, facecolor="white", bbox_inches="tight")
    fig.savefig(OUT / "Figure_2_human_bootstrap_band.pdf",
                facecolor="white", bbox_inches="tight")
    plt.show()
else:
    print("Run Notebook 3 to generate normalized profiles and human bootstrap CIs for Figure 2.")


## Figure 4: Iraq-excluded 65-country comparisons

Uses the **same country-specific normalized gradients** computed by Notebook 3 and excludes Iraq from the primary visualization. The full 66-country comparison remains a sensitivity analysis. In contrast to the old figure code, this version labels only selected extreme-gradient observations to avoid overlap. All points remain plotted; cross-country OLS lines are unchanged.

In [ ]:
grad_path = LOCKED / "08d_country_gradient_wide.csv"
if grad_path.exists():
    gradient_data = pd.read_csv(grad_path)
    assert "country_name" in gradient_data and "Human" in gradient_data
    main = gradient_data.loc[
        ~gradient_data["country_name"].astype(str).str.strip().str.casefold().eq("iraq")
    ].copy()
    assert len(main) == 65
    fig, axes = plt.subplots(3, 2, figsize=(10.2, 13.4))
    for letter, name, ax in zip("ABCDEF", SOURCE_ORDER[1:], axes.flat):
        d = main[["country_name", "Human", name]].dropna()
        x = d["Human"].to_numpy(float)
        y = d[name].to_numpy(float)
        m, b = np.polyfit(x, y, 1)
        low = float(min(np.min(x), np.min(y)))
        high = float(max(np.max(x), np.max(y)))
        pad = 0.06 * (high - low)
        low, high = low - pad, high + pad
        xx = np.linspace(low, high, 100)
        ax.scatter(x, y, s=14, alpha=0.8)
        ax.plot(xx, xx, linestyle="--", color="0.65", linewidth=1)
        ax.plot(xx, m * xx + b, color="0.2", linewidth=1.3)
        ax.set(xlim=(low, high), ylim=(low, high),
               xlabel="Human normalized gradient",
               ylabel="LLM normalized gradient")
        ax.set_title(
            letter + ". " + name + "\nPearson r = " +
            format(pearsonr(x, y).statistic, ".2f") +
            "; Spearman ρ = " + format(spearmanr(x, y).statistic, ".2f") +
            "; N = " + str(len(d)),
            fontsize=10)
        # Label only the two most negative and two most positive human gradients.
        special = np.unique(np.r_[np.argsort(x)[:2], np.argsort(x)[-2:]])
        for i in special:
            ax.annotate(str(d.iloc[i]["country_name"]), (x[i], y[i]),
                        xytext=(3, 3), textcoords="offset points", fontsize=6)
    fig.tight_layout(h_pad=2.2, w_pad=2)
    fig.savefig(OUT / "Figure_4_65_countries.png",
                dpi=300, facecolor="white", bbox_inches="tight")
    fig.savefig(OUT / "Figure_4_65_countries.pdf",
                facecolor="white", bbox_inches="tight")
    plt.show()
else:
    print("Run Notebook 3 to generate 08d_country_gradient_wide.csv for Figure 4.")


## Reproducibility notes

- These additional analyses are post-preregistration; no claim of confirmatory status is made.
- Figure 2 confidence intervals use stored country-cluster bootstrap results from Notebook 3. Re-running Notebook 3 is necessary to regenerate them if not present.
- Table S8 recalculates human income-group normalization weights in each sample; it therefore measures the **pooled** profile, not the country-fixed-effect slope.
- The updated manuscript Figure 4 is a presentation of the Iraq-excluded 65-country analysis; Iraq remains in the pooled primary analyses. No observation is silently removed.
- OLS/Lasso accuracy is optional and only computed if their existing **out-of-sample predictions** are present from Notebook 5.
- Original model predictions and WVS microdata are intentionally not included in this repository.